In [ ]:
import pandas as pd
import numpy as np
# === Load CSV ===
df = pd.read_csv("/home/gzu5140/Keerthana_b1042/grnInference/plot_data/parameter_scan/A_to_B_inference_results.csv", index_col=0)
df['ratio'] = df['k_add_gene_1_to_gene_2']/df['k_on']

In [ ]:
df

In [ ]:

# --- parameter columns ---
gene_specific = [
    "k_on", "k_off", "mrna_half_life", "protein_half_life",
    "k_prod_mRNA", "k_prod_protein", "pi_on", "ratio"
]
pair_specific = [
    "k_add_gene_1_to_gene_2", "n_gene_1_to_gene_2"
]

# Split into gene 1 and gene 2
g1 = df[df["gene_id"] == 1][["pair_id"] + gene_specific + pair_specific].copy()
g2 = df[df["gene_id"] == 2][["pair_id"] + gene_specific].copy()

# Rename columns for clarity
g1 = g1.rename(columns={c: f"{c}_g1" for c in gene_specific})
g2 = g2.rename(columns={c: f"{c}_g2" for c in gene_specific})

# Merge by pair_id
merged = pd.merge(g1, g2, on="pair_id", suffixes=("", "_drop"))
merged = merged[[c for c in merged.columns if not c.endswith("_drop")]]  # clean duplicates

# --- keep shared parameters only once ---
merged = merged.loc[:, ~merged.columns.duplicated()]

# Example label: significant if directed p-value < 0.01
pvals = (
    df[df["gene_id"] == 1][["pair_id", "corr_gene_gene_gene_1_mRNA_gene_2_mRNA_t1_t20"]]
    .drop_duplicates("pair_id")
)
merged = merged.merge(pvals, on="pair_id", how="left")
merged["significant"] = (merged["corr_gene_gene_gene_1_mRNA_gene_2_mRNA_t1_t20"] < 0.01).astype(int)


In [ ]:
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

X = merged.drop(columns=["pair_id", "significant", "corr_gene_gene_gene_1_mRNA_gene_2_mRNA_t1_t20", "ratio_g1"])
y = merged["significant"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

model = xgb.XGBClassifier(
    n_estimators=200,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    eval_metric="logloss"
)
model.fit(X_train, y_train)

print(classification_report(y_test, model.predict(X_test)))


In [ ]:
import shap
booster = model.get_booster()
explainer = shap.TreeExplainer(booster)
shap_values = explainer.shap_values(X)
shap.summary_plot(shap_values, X)


In [ ]:
import shap
explainer = shap.explainers.Tree(model.get_booster())
shap_values = explainer(X)
shap.plots.bar(shap_values)


In [ ]:
# %% Imports
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

X = merged.drop(columns=["pair_id", "significant", "corr_gene_gene_gene_1_mRNA_gene_2_mRNA_t1_t20", "ratio_g1"])
y = merged["significant"]


# %% Split and fit
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

rf = RandomForestClassifier(
    n_estimators=300,
    max_depth=None,
    min_samples_split=5,
    min_samples_leaf=3,
    random_state=42,
    n_jobs=-1,
    class_weight="balanced_subsample"
)
rf.fit(X_train, y_train)

# %% Evaluate
y_pred = rf.predict(X_test)
print(classification_report(y_test, y_pred))
sns.heatmap(confusion_matrix(y_test, y_pred), annot=True, fmt="d", cmap="coolwarm")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()

# %% Feature importance (mean decrease impurity)
importances = pd.DataFrame({
    "Feature": X.columns,
    "Importance": rf.feature_importances_
}).sort_values("Importance", ascending=False)

plt.figure(figsize=(7,6))
sns.barplot(data=importances.head(20), x="Importance", y="Feature", palette="Reds_r")
plt.title("Random Forest Feature Importance (MDI)")
plt.tight_layout()
plt.show()


In [ ]:
import numpy as np
import pandas as pd

def count_feature_depths_rf(rf_model, feature_names, max_depth=2):
    """
    Count how often each feature is used in the top few levels (<= max_depth)
    of every tree in a scikit-learn RandomForest.
    Lower depth = 'easier' / more decisive feature.
    """
    counts = {name: 0 for name in feature_names}

    for est in rf_model.estimators_:
        tree = est.tree_
        features = tree.feature
        children_left, children_right = tree.children_left, tree.children_right

        # --- compute depth of each node using BFS ---
        node_depth = np.zeros(tree.node_count, dtype=int)
        stack = [(0, 0)]  # (node_id, depth)
        while stack:
            node_id, depth = stack.pop()
            node_depth[node_id] = depth
            left, right = children_left[node_id], children_right[node_id]
            if left != -1 and right != -1:  # not a leaf
                stack.append((left, depth + 1))
                stack.append((right, depth + 1))

        # --- count features used in top layers ---
        for node_id, f_idx in enumerate(features):
            if f_idx >= 0 and node_depth[node_id] <= max_depth:
                counts[feature_names[f_idx]] += 1

    df = pd.DataFrame.from_dict(counts, orient="index", columns=["count"])
    df["fraction"] = df["count"] / df["count"].sum()
    df = df.sort_values("count", ascending=False)
    return df

# Example usage
df_top = count_feature_depths_rf(rf, X.columns, max_depth=3)
print(df_top.head(10))


In [ ]:
import shap
explainer = shap.TreeExplainer(rf)
shap_values = explainer.shap_values(X)
shap.summary_plot(shap_values[1], X, plot_type="bar")
